# Fungsi Preprocessing pada data raw

## 1. Mengimpor Library dan module yang dibutuhkan

WAJIB ADA KARENA INI NOTEBOOK

In [1]:
# LIBRARY
import sys
sys.path.append('..')
# Menambahkan path ke folder parent (satu level di atas)
# Tujuannya agar bisa import module dari folder src/
# (karena notebook ada di folder notebooks/)

# LIBRARY DAN MODULE DARI FILE LAIN
import pandas as pd
from sklearn.model_selection import train_test_split
from src import bersihkan_teks_total,bersihkan_teks_minimal,encode_label_series
from src import simpan_objek

## 2. Logika Code

### 2.1 Load dataset

In [6]:
# ============================================================
# 🔥 UNDERSAMPLING
# ============================================================

df_hoax = df[df['label'] == 1]
df_fakta = df[df['label'] == 0]

# Ambil sebagian hoax (sesuaikan)
df_hoax_sample = df_hoax.sample(n=1000, random_state=42)

# Gabungkan
df_balanced = pd.concat([df_hoax_sample, df_fakta])

# Shuffle
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print("\nDistribusi setelah undersampling:")
print(df_balanced['label'].value_counts())


Distribusi setelah undersampling:
label
1.0    1000
0.0     730
Name: count, dtype: int64


### 2.2 Split Data (80% training, 20% testing)

In [3]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['tweet'],        # Fitur (teks)
    df['label'],        # Target (label hoaks / bukan)
    test_size=0.2,      # 20% data untuk testing
    random_state=42,    # Seed agar hasil split selalu sama (reproducible)
    stratify=df['label'] # Menjaga proporsi label tetap seimbang di train & test
)

### 2.3 Proses Pembersihan teks 

In [4]:
X_train_clean = X_train_raw.apply(bersihkan_teks_total)
# Membersihkan data training menggunakan preprocessing lengkap
# apply() → menerapkan fungsi ke setiap baris data teks

# SELANJUTNYA MEMBERSIHKAN DATASET DENGAN 2 TIPE
X_test_clean = X_test_raw.apply(bersihkan_teks_total)  
# Data uji versi bersih (ideal condition)

X_test_noisy = X_test_raw.apply(bersihkan_teks_minimal)  
# Data uji versi noisy (real-world condition)
# Digunakan untuk menguji robustness model

### 2.4 Membuat dictionary dan menyimpan hasil preprocessing

In [5]:
data_mentah = {
    'X_train_clean': X_train_clean,  # Data training yang sudah dibersihkan
    'y_train': y_train,              # Label training

    'X_test_clean': X_test_clean,    # Data test bersih
    'y_test': y_test,                # Label test

    'X_test_noisy': X_test_noisy     # Data test kotor (simulasi dunia nyata)
}

# Menyimpan dictionary ke file .pkl menggunakan pickle
simpan_objek(data_mentah, '../data/processed/data_teks_bersih.pkl')

print("Data teks berhasil dibersihkan dan disimpan!")
# Notifikasi bahwa proses selesai

Data teks berhasil dibersihkan dan disimpan!
